In [29]:
import numpy as np
import pandas as pd

def get_model_summary(model, input_shape=None, model_name=""):
    """
    Prints a summary of the given ML model based on its type.
    - For sklearn: Prints parameters and feature importances if available.
    - For TensorFlow/Keras: Prints the model summary.
    - For PyTorch: Prints the model architecture.
    """
    model_name = model_name or type(model).__name__
    print(model_name)

    try:
        # Scikit-learn models
        from sklearn.base import BaseEstimator
        if isinstance(model, BaseEstimator):
            print(f"Model: {model_name}")
            print("Parameters:")
            print(model.get_params())
            if hasattr(model, "feature_importances_"):
                print("\nFeature Importances:")
                print(model.feature_importances_)
            elif hasattr(model, "coef_"):
                print("\nModel Coefficients:")
                print(model.coef_)
            return

    except ImportError:
        pass

    try:
        # TensorFlow/Keras models
        from tensorflow.keras import Model
        if isinstance(model, Model):
            print(f"Model: {model_name}")
            input_shape = model.input_shape
            if input_shape:
                print("\nKeras Model Summary:")
                #model.build(input_shape)
                model.summary()
            else:
                print("Provide `input_shape` to build and summarize the model.")
            return
    except ImportError:
        pass

    try:
        # PyTorch models
        import torch.nn as nn
        from torchsummary import summary
        
        first_layer = next(model.children())
        if isinstance(first_layer, nn.Linear):
            input_size = first_layer.in_features
            print("Input size:", input_size)
            summary(model, input_size=input_size)
        elif isinstance(first_layer, nn.Conv2d):
            print("First layer expects Conv2D input. Check your specific dimensions.")
        
            return
    except ImportError:
        pass

    print(f"Model type {model_name} not supported or library not installed.")


In [19]:
import os
import h5py
import torch
import onnx
import joblib

def detect_framework(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    if extension in [".h5", ".keras"]:
        return "Keras"


    elif extension in [".pb", ""]:
        if os.path.exists(os.path.join(file_path, "saved_model.pb")):
            return "TensorFlow (SavedModel)"
    
    elif extension in [".pt", ".pth"]:
        try:
            _ = torch.load(file_path)
            return "PyTorch"
        except Exception:
            pass

    elif extension == ".onnx":
        try:
            _ = onnx.load(file_path)
            return "ONNX"
        except Exception:
            pass

    elif extension in [".joblib", ".pkl"]:
        try:
            _ = joblib.load(file_path)
            return "Scikit-learn"
        except Exception:
            pass

    return "Unknown framework"

# Test the function
file_path = "model.h5"  # Replace with your file path
framework = detect_framework(file_path)
print(f"Detected framework: {framework}")


Detected framework: Keras


In [23]:
from tensorflow.keras.models import load_model
model_path = "./models/nasa_FD001_lstm.keras"
print(detect_framework(model_path))
keras_model = load_model(model_path)
get_model_summary(keras_model)

Keras
Model: Sequential

Keras Model Summary:
Model: "sequential_13"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d_13 (Conv1D)          (None, 18, 64)            3136      
                                                                 
 lstm_13 (LSTM)              (None, 128)               98816     
                                                                 
 dense_13 (Dense)            (None, 1)                 129       
                                                                 
Total params: 102081 (398.75 KB)
Trainable params: 102081 (398.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [22]:
import pickle
model_path = "./models/linear_regression.pkl"
print(detect_framework(model_path))
with open(model_path, "rb") as file:
    loaded_model = pickle.load(file)
    get_model_summary(loaded_model)

Scikit-learn
Model: LinearRegression
Parameters:
{'copy_X': True, 'fit_intercept': True, 'n_jobs': None, 'positive': False}

Model Coefficients:
[[0.04753664]]


In [31]:
model_path = "./models/pytorch_model.bin"
print(detect_framework(model_path))
import torch

# Load the entire model from the .bin file
from transformers import AutoModel

# Load the Hugging Face model from the .bin file
model = AutoModel.from_pretrained(model_path)
get_model_summary(model)

Unknown framework


OSError: Incorrect path_or_model_id: './models/pytorch_model.bin'. Please provide either the path to a local folder or the repo_id of a model on the Hub.